# AE architecture sweep

Stage 1 (capacity, 12 configs) and Stage 2 (optimisation, 9 configs at
the Stage-1 winner) at fixed latent dim 16 on the block-stride 80/10/10
split. Picks the winner by validation MSE-z and persists the trained
winning model to `outputs/checkpoints/03_ae_arch_sweep/ae_sweep_winner.pt`.

Per-stage results are logged to `outputs/csvs/03_ae_arch_sweep/stage1.csv` and
`outputs/csvs/03_ae_arch_sweep/stage2.csv` incrementally: each row is written as
soon as the config finishes, so an interrupt mid-stage keeps the rows
already completed.

In [1]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Anchor to the repo root so paleoreco imports and relative file paths
# resolve regardless of where Jupyter was launched from.
REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from paleoreco.data import (
    PaleoFieldDataset,
    apply_zscore,
    build_prior_cube,
    compute_zscore_stats,
)
from paleoreco.splits import block_stride_split, summarize_split
from paleoreco.models.autoencoder import ConvAE, count_parameters
from paleoreco.train_ae import set_seed, train
from paleoreco.eval.ae import reconstruct_split
from paleoreco.eval.shared import compute_split_metrics

/vol/bitbucket/egg25/diss/sparse_paleoclimate_field_reconstruction/paleo/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
LATENT_DIM = 16
MAX_EPOCHS = 250
PATIENCE = 25
BATCH_SIZE = 32

# Stage 1: capacity grid at the v1-default optimiser settings.
BASE_CHANNELS = (8, 16, 32, 64)
DEPTHS = (2, 3, 4)
LR_STAGE1 = 1e-3
WD_STAGE1 = 1e-4

# Stage 2: optimiser grid at the Stage-1 winning (base, depth).
LRS = (3e-4, 1e-3, 3e-3)
WDS = (1e-5, 1e-4, 1e-3)

OUT_DIR = Path(REPO_ROOT) / "outputs"
CSV_DIR = OUT_DIR / "csvs" / "03_ae_arch_sweep"
CKPT_DIR = OUT_DIR / "checkpoints" / "03_ae_arch_sweep"
STAGE1_CSV = CSV_DIR / "stage1.csv"
STAGE2_CSV = CSV_DIR / "stage2.csv"
WINNER_PT = CKPT_DIR / "ae_sweep_winner.pt"

OUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

## Data and split

In [3]:
data = build_prior_cube(
    prior_csv=os.path.join(REPO_ROOT, "data/Prior.csv"),
    cache_path=os.path.join(REPO_ROOT, "data/cache/prior_cube.npz"),
)
cube = data["cube"]
ages = data["ages"]
valid = data["valid"]

split = block_stride_split(len(ages))
print(summarize_split(ages, split))

stats = compute_zscore_stats(cube, split["train"], valid)
cube_z = apply_zscore(cube, stats)
mask = stats["safe_valid"]
print(f"\nsafe_valid cells: {int(mask.sum())} / {mask.size}")

train:  644 ages, range [30100, 49175] yr BP [DO5=37, DO6=29, DO7=36, DO8=36, DO9=25, DO10=36, DO11=36, DO12=36, between=373]
  val:   80 ages, range [34100, 45075] yr BP [DO6=7, between=73]
 test:   80 ages, range [29100, 40075] yr BP [DO9=11, between=69]

safe_valid cells: 2048 / 2048


In [4]:
train_dataset = PaleoFieldDataset(cube_z, mask, split["train"])
val_dataset = PaleoFieldDataset(cube_z, mask, split["val"])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"train batches: {len(train_loader)}  val batches: {len(val_loader)}")

device: cuda
train batches: 21  val batches: 3


## Per-config training helper

In [5]:
def train_one_config(
    base_channels: int,
    depth: int,
    lr: float,
    weight_decay: float,
) -> dict:
    """Run one training config end-to-end and return its evaluation bundle.

    Sets the seed before model construction so model init is deterministic;
    train() reseeds before its own data-loader iteration so DataLoader
    shuffling is the same across configs.
    """
    set_seed(SEED)
    model = ConvAE(
        latent_dim=LATENT_DIM,
        base_channels=base_channels,
        depth=depth,
    )
    n_params = count_parameters(model)

    t0 = time.perf_counter()
    out = train(
        model,
        train_loader,
        val_loader,
        mask=mask,
        zscore_std=stats["std"],
        lr=lr,
        weight_decay=weight_decay,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        device=device,
        seed=SEED,
        verbose=False,
        progress=True,
    )
    seconds = time.perf_counter() - t0

    # Load best-val weights into the trained model and re-evaluate val to
    # get the full metric bundle (train() history tracks only mse_z / rmse_z).
    model.load_state_dict(out["best_state_dict"])
    truth_z, pred_z = reconstruct_split(
        model, val_dataset, device=device, batch_size=BATCH_SIZE,
    )
    val_metrics = compute_split_metrics(truth_z, pred_z, mask)

    return {
        "state_dict": out["best_state_dict"],
        "val_metrics": val_metrics,
        "n_params": n_params,
        "epochs_trained": out["epochs_trained"],
        "best_epoch": out["best_epoch"],
        "seconds": seconds,
    }

## Stage 1: capacity sweep (12 configs)

Fix `latent_dim=16`, `lr=1e-3`, `weight_decay=1e-4`. Sweep
`base_channels` × `depth` = 4 × 3 grid.

In [6]:
stage1_rows: list[dict] = []
stage1_configs = [(b, d) for b in BASE_CHANNELS for d in DEPTHS]

for i, (base, depth) in enumerate(stage1_configs, start=1):
    print(f"\n[Stage 1 {i:>2}/{len(stage1_configs)}] base={base}  depth={depth}")
    result = train_one_config(
        base_channels=base, depth=depth,
        lr=LR_STAGE1, weight_decay=WD_STAGE1,
    )
    m = result["val_metrics"]
    row = {
        "base_channels":  base,
        "depth":          depth,
        "val_mse_z":      m["mse_z"],
        "val_rmse_z":     m["rmse_z"],
        "val_rrmse":      m["rrmse"],
        "val_r_squared":  m["r_squared"],
        "val_E_d":        m["E_d"],
        "n_params":       result["n_params"],
        "epochs_trained": result["epochs_trained"],
        "best_epoch":     result["best_epoch"],
    }
    stage1_rows.append(row)
    pd.DataFrame(stage1_rows).to_csv(STAGE1_CSV, index=False)
    print(
        f"  val_mse_z={m['mse_z']:.4f}  R²={m['r_squared']:.3f}  "
        f"E_d={m['E_d']:.3f}  params={result['n_params']:,}  "
        f"epochs={result['epochs_trained']}  ({result['seconds']:.0f}s)"
    )


[Stage 1  1/12] base=8  depth=2


train(d=16):  96%|█████████▋| 241/250 [00:41<00:01,  5.78ep/s, val_mse_z=0.0887, best=0.0886, s/ep=0.2]


  val_mse_z=0.0886  R²=0.895  E_d=0.893  params=70,590  epochs=242  (47s)

[Stage 1  2/12] base=8  depth=3


train(d=16):  93%|█████████▎| 232/250 [00:51<00:03,  4.52ep/s, val_mse_z=0.0921, best=0.0920, s/ep=0.2]


  val_mse_z=0.0920  R²=0.890  E_d=0.889  params=46,158  epochs=233  (51s)

[Stage 1  3/12] base=8  depth=4


...

  val_mse_z=0.0626  R²=0.926  E_d=0.924  params=709,746  epochs=110  (21s)

[Stage 1 11/12] base=64  depth=3


train(d=16):  31%|███       | 77/250 [00:18<00:41,  4.18ep/s, val_mse_z=0.0698, best=0.0660, s/ep=0.2]


  val_mse_z=0.0660  R²=0.921  E_d=0.920  params=1,030,386  epochs=78  (18s)

[Stage 1 12/12] base=64  depth=4


train(d=16):  40%|████      | 101/250 [00:29<00:44,  3.38ep/s, val_mse_z=0.0765, best=0.0703, s/ep=0.3]

  val_mse_z=0.0703  R²=0.916  E_d=0.915  params=3,256,818  epochs=102  (30s)


In [7]:
stage1_df = pd.DataFrame(stage1_rows).sort_values("val_mse_z").reset_index(drop=True)
print(stage1_df.to_string(index=False))

winner_stage1 = stage1_df.iloc[0]
BASE_STAR = int(winner_stage1["base_channels"])
DEPTH_STAR = int(winner_stage1["depth"])
print(
    f"\nStage-1 winner: base={BASE_STAR}  depth={DEPTH_STAR}  "
    f"val_mse_z={winner_stage1['val_mse_z']:.4f}"
)

 base_channels  depth  val_mse_z  val_rmse_z  val_rrmse  val_r_squared  val_E_d  n_params  epochs_trained  best_epoch
            64      2   0.062557    0.250115   0.272906       0.925522 0.924402    709746             110          84
            32      2   0.062868    0.250734   0.273582       0.925153 0.923914    313410             220         194
            32      3   0.065656    0.256234   0.279583       0.921833 0.920648    326274             121          95
            64      3   0.065996    0.256896   0.280305       0.921429 0.920328   1030386              78          52
            16      3   0.067735    0.260260   0.283976       0.919358 0.918235    115914             250         235
            16      2   0.069266    0.263184   0.287166       0.917536 0.916626    146346             250         232
            64      4   0.070271    0.265087   0.289242       0.916339 0.915289   3256818             102          76
            32      4   0.070666    0.265830   0.290054 

## Stage 2: optimisation sweep at the Stage-1 winner (9 configs)

Fix `(base, depth)` at `(BASE_STAR, DEPTH_STAR)`. Sweep `lr` ×
`weight_decay` = 3 × 3 grid. The `(lr=1e-3, weight_decay=1e-4)` cell
duplicates one Stage-1 run; it is re-trained for loop uniformity.

In [8]:
stage2_rows: list[dict] = []
stage2_states: dict[tuple[float, float], dict] = {}
stage2_configs = [(lr, wd) for lr in LRS for wd in WDS]

for i, (lr, wd) in enumerate(stage2_configs, start=1):
    print(
        f"\n[Stage 2 {i:>2}/{len(stage2_configs)}] "
        f"base={BASE_STAR}  depth={DEPTH_STAR}  "
        f"lr={lr:.0e}  wd={wd:.0e}"
    )
    result = train_one_config(
        base_channels=BASE_STAR, depth=DEPTH_STAR,
        lr=lr, weight_decay=wd,
    )
    m = result["val_metrics"]
    row = {
        "lr":             lr,
        "weight_decay":   wd,
        "val_mse_z":      m["mse_z"],
        "val_rmse_z":     m["rmse_z"],
        "val_rrmse":      m["rrmse"],
        "val_r_squared":  m["r_squared"],
        "val_E_d":        m["E_d"],
        "n_params":       result["n_params"],
        "epochs_trained": result["epochs_trained"],
        "best_epoch":     result["best_epoch"],
    }
    stage2_rows.append(row)
    stage2_states[(lr, wd)] = {
        "state_dict": result["state_dict"],
        "val_mse_z":  m["mse_z"],
        "best_epoch": result["best_epoch"],
    }
    pd.DataFrame(stage2_rows).to_csv(STAGE2_CSV, index=False)
    print(
        f"  val_mse_z={m['mse_z']:.4f}  R²={m['r_squared']:.3f}  "
        f"E_d={m['E_d']:.3f}  epochs={result['epochs_trained']}  "
        f"({result['seconds']:.0f}s)"
    )


[Stage 2  1/9] base=64  depth=2  lr=3e-04  wd=1e-05


train(d=16):  71%|███████   | 177/250 [00:32<00:13,  5.40ep/s, val_mse_z=0.0615, best=0.0612, s/ep=0.2]


  val_mse_z=0.0612  R²=0.927  E_d=0.926  epochs=178  (33s)

[Stage 2  2/9] base=64  depth=2  lr=3e-04  wd=1e-04


train(d=16):  71%|███████   | 177/250 [00:32<00:13,  5.41ep/s, val_mse_z=0.0614, best=0.0612, s/ep=0.2]


  val_mse_z=0.0612  R²=0.927  E_d=0.926  epochs=178  (33s)

[Stage 2  3/9] base=64  depth=2  lr=3e-04  wd=1e-03


...

  val_mse_z=0.0642  R²=0.924  E_d=0.922  epochs=93  (16s)

[Stage 2  8/9] base=64  depth=2  lr=3e-03  wd=1e-04


train(d=16):  37%|███▋      | 92/250 [00:17<00:29,  5.34ep/s, val_mse_z=0.0670, best=0.0642, s/ep=0.2]


  val_mse_z=0.0642  R²=0.924  E_d=0.922  epochs=93  (17s)

[Stage 2  9/9] base=64  depth=2  lr=3e-03  wd=1e-03


train(d=16):  37%|███▋      | 92/250 [00:17<00:30,  5.23ep/s, val_mse_z=0.0688, best=0.0643, s/ep=0.2]


  val_mse_z=0.0643  R²=0.923  E_d=0.922  epochs=93  (18s)


In [9]:
stage2_df = pd.DataFrame(stage2_rows).sort_values("val_mse_z").reset_index(drop=True)
print(stage2_df.to_string(index=False))

winner_stage2 = stage2_df.iloc[0]
LR_STAR = float(winner_stage2["lr"])
WD_STAR = float(winner_stage2["weight_decay"])
WINNER_STATE = stage2_states[(LR_STAR, WD_STAR)]

print(
    f"\nFinal winner: base={BASE_STAR}  depth={DEPTH_STAR}  "
    f"lr={LR_STAR:.0e}  wd={WD_STAR:.0e}\n"
    f"  val_mse_z={WINNER_STATE['val_mse_z']:.4f}  "
    f"best_epoch={WINNER_STATE['best_epoch']}"
)

    lr  weight_decay  val_mse_z  val_rmse_z  val_rrmse  val_r_squared  val_E_d  n_params  epochs_trained  best_epoch
0.0003       0.00100   0.061227    0.247442   0.269989       0.927106 0.925883    709746             178         152
0.0003       0.00010   0.061228    0.247444   0.269992       0.927105 0.925882    709746             178         152
0.0003       0.00001   0.061229    0.247445   0.269993       0.927104 0.925881    709746             178         152
0.0010       0.00010   0.062556    0.250112   0.272903       0.925524 0.924403    709746             110          84
0.0010       0.00001   0.062558    0.250116   0.272908       0.925521 0.924401    709746             110          84
0.0010       0.00100   0.062559    0.250118   0.272910       0.925520 0.924399    709746             110          84
0.0030       0.00010   0.064231    0.253437   0.276531       0.923530 0.922345    709746              93          67
0.0030       0.00001   0.064240    0.253455   0.276551       0.9

## Persist winner

Writes `outputs/checkpoints/03_ae_arch_sweep/ae_sweep_winner.pt` with the trained
state\_dict and the config + training metadata. This is the only
checkpoint produced by the sweep.

In [10]:
checkpoint = {
    "state_dict": WINNER_STATE["state_dict"],
    "config": {
        "latent_dim":    LATENT_DIM,
        "base_channels": BASE_STAR,
        "depth":         DEPTH_STAR,
        "in_channels":   3,
        "out_channels":  2,
        "grid_shape":    (32, 64),
    },
    "training": {
        "lr":           LR_STAR,
        "weight_decay": WD_STAR,
        "max_epochs":   MAX_EPOCHS,
        "patience":     PATIENCE,
        "seed":         SEED,
        "split":        "block_stride_80_10_10",
        "monitor":      "val_mse_z",
    },
    "val_mse_z_at_early_stop": WINNER_STATE["val_mse_z"],
    "best_epoch":              WINNER_STATE["best_epoch"],
}
torch.save(checkpoint, WINNER_PT)
print(f"saved {WINNER_PT}")
print(f"  config:   {checkpoint['config']}")
print(f"  training: {checkpoint['training']}")
print(f"  val_mse_z_at_early_stop: {checkpoint['val_mse_z_at_early_stop']:.4f}")
print(f"  best_epoch: {checkpoint['best_epoch']}")

saved /vol/bitbucket/egg25/diss/sparse_paleoclimate_field_reconstruction/outputs/checkpoints/ae_sweep_winner.pt
  config:   {'latent_dim': 16, 'base_channels': 64, 'depth': 2, 'in_channels': 3, 'out_channels': 2, 'grid_shape': (32, 64)}
  training: {'lr': 0.0003, 'weight_decay': 0.001, 'max_epochs': 250, 'patience': 25, 'seed': 42, 'split': 'block_stride_80_10_10', 'monitor': 'val_mse_z'}
  val_mse_z_at_early_stop: 0.0612
  best_epoch: 152
